In [2]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('/content/loan_approval_dataset.csv')

In [5]:
df.columns = df.columns.str.strip()

In [6]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

In [7]:
df.drop_duplicates(inplace=True)
print(df.isnull().sum())

loan_id                     0
no_of_dependents            0
education                   0
self_employed               0
income_annum                0
loan_amount                 0
loan_term                   0
cibil_score                 0
residential_assets_value    0
commercial_assets_value     0
luxury_assets_value         0
bank_asset_value            0
loan_status                 0
dtype: int64


In [8]:
df["loan_status"] = df["loan_status"].map({"Approved": 1, "Rejected": 0})  # standardize target column

In [16]:
df["loan_status"].value_counts()

,count
loan_status,
1,2656
0,1613


In [9]:
df["total_assets"] = (
    df["residential_assets_value"] + df["commercial_assets_value"] +
    df["luxury_assets_value"] + df["bank_asset_value"]
)
#total assets

In [10]:
df["loan_to_income"] = df["loan_amount"] / df["income_annum"] # debt-to-income style ratio

In [11]:
approval_rate = df["loan_status"].mean() * 100
print(f"Overall approval rate: {approval_rate:.2f}%") #Overall approval rate

Overall approval rate: 62.22%


In [12]:
df["cibil_band"] = pd.cut(df["cibil_score"], bins=[0,600,700,750,900],
                           labels=["<600","600-700","700-750","750+"])
print(df.groupby("cibil_band")["loan_status"].mean() * 100)
#Approval rate by credit score band

cibil_band
<600       25.279851
600-700    99.428571
700-750    99.731183
750+       99.430199
Name: loan_status, dtype: float64


/tmp/ipykernel_2317/3635934028.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("cibil_band")["loan_status"].mean() * 100)


In [13]:
print(df.groupby(["education","self_employed"])["loan_status"].mean() * 100)
#Approval rate by education and self-employed status

education     self_employed
Graduate      No               62.534435
              Yes              62.369668
Not Graduate  No               61.844660
              Yes              62.100457
Name: loan_status, dtype: float64


In [14]:
df["lti_band"] = pd.cut(df["loan_to_income"], bins=[0,1,2,3,10],
                         labels=["<1x","1-2x","2-3x","3x+"])
print(df.groupby("lti_band")["loan_status"].mean() * 100)
#Approval rate by loan-to-income ratio band

lti_band
<1x           NaN
1-2x    53.333333
2-3x    58.385093
3x+     66.634241
Name: loan_status, dtype: float64


/tmp/ipykernel_2317/1040206876.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("lti_band")["loan_status"].mean() * 100)


In [15]:
df.to_csv("loan_cleaned.csv", index=False)

In [ ]:
from google.colab import files
files.download("loan_cleaned.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sqlite3

In [ ]:
conn = sqlite3.connect('loan_cleaned.db')
df.to_sql('loan_cleaned', conn, if_exists='replace', index=False)

4269

In [ ]:
query = """
SELECT
  CASE
    WHEN cibil_score < 600 THEN '<600'
    WHEN cibil_score < 700 THEN '600-700'
    WHEN cibil_score < 750 THEN '700-750'
    ELSE '750+'
  END AS cibil_band,
  COUNT(*) AS total_applications,
  SUM(CASE WHEN loan_status = 1 THEN 1 ELSE 0 END) AS approved,
  ROUND(100.0 * SUM(CASE WHEN loan_status = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS approval_rate_pct
FROM loan_cleaned
GROUP BY cibil_band
ORDER BY approval_rate_pct DESC;
"""
pd.read_sql(query, conn)
#Approval rate by credit score band

,cibil_band,total_applications,approved,approval_rate_pct
0,700-750,372,371,99.73
1,750+,1053,1047,99.43
2,600-700,700,696,99.43
3,<600,2144,542,25.28


In [ ]:
query = """
SELECT education, self_employed,
       ROUND(100.0 * SUM(CASE WHEN loan_status=1 THEN 1 ELSE 0 END)/COUNT(*),2) AS approval_rate_pct
FROM loan_cleaned
GROUP BY education, self_employed;
"""
pd.read_sql(query, conn)
#Approval rate by education and self-employment

,education,self_employed,approval_rate_pct
0,Graduate,No,62.53
1,Graduate,Yes,62.37
2,Not Graduate,No,61.84
3,Not Graduate,Yes,62.10


In [ ]:
query = """
SELECT loan_status,
       ROUND(AVG(income_annum),2) AS avg_income,
       ROUND(AVG(loan_amount),2) AS avg_loan_amount,
       ROUND(AVG(cibil_score),2) AS avg_cibil_score
FROM loan_cleaned
GROUP BY loan_status;
"""
pd.read_sql(query, conn)
#Average loan amount and income for approved vs rejected

,loan_status,avg_income,avg_loan_amount,avg_cibil_score
0,0,5113825.17,14946063.24,429.47
1,1,5025903.61,15247251.51,703.46


In [ ]:
query = """
SELECT COUNT(*) AS high_risk_applicants,
       ROUND(100.0 * SUM(CASE WHEN loan_status=1 THEN 1 ELSE 0 END)/COUNT(*),2) AS approval_rate_pct
FROM loan_cleaned
WHERE (loan_amount / income_annum) > 3
  AND (residential_assets_value + commercial_assets_value + luxury_assets_value + bank_asset_value) < 1000000;
"""
pd.read_sql(query, conn)
#High risk applicants


,high_risk_applicants,approval_rate_pct
0,3,33.33
